In [ ]:
# app/llm/base.py

from typing import Protocol, TypeVar
from pydantic import BaseModel

T = TypeVar("T", bound=BaseModel)


class StructuredGenerator(Protocol):
    async def generate(
        self,
        *,
        system_prompt: str,
        user_prompt: str,
        response_model: type[T],
    ) -> T:
        ...

In [ ]:
# app/services/dispute_classifier.py

from app.llm.base import StructuredGenerator
from app.loaders.prompt_loader import PromptLoader
from app.loaders.category_loader import CategoryLoader
from app.schemas.card_network import CardNetwork
from app.schemas.language import Language
from app.schemas.classification import ClassificationResult


class DisputeClassifier:

    def __init__(
        self,
        *,
        model: StructuredGenerator,
        prompt_loader: PromptLoader,
        category_loader: CategoryLoader,
    ):
        self._model = model
        self._prompt_loader = prompt_loader
        self._category_loader = category_loader

    async def classify(
        self,
        *,
        case_context: str,
        network: CardNetwork,
        language: Language,
    ) -> ClassificationResult:

        system_prompt = self._prompt_loader.load_system(
            prompt_type="classification",
        )

        task_template = self._prompt_loader.load_task(
            prompt_type="classification",
            network=network,
            language=language,
        )

        category_text = self._category_loader.as_prompt_text(
            network=network,
            language=language,
        )

        user_prompt = task_template.format(
            case_context=case_context,
            category_descriptions=category_text,
        )

        return await self._model.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            response_model=ClassificationResult,
        )

In [ ]:
# app/services/dispute_reasoner.py

from app.llm.base import StructuredGenerator
from app.loaders.prompt_loader import PromptLoader
from app.loaders.category_loader import CategoryLoader
from app.schemas.card_network import CardNetwork
from app.schemas.language import Language
from app.schemas.classification import ClassificationResult
from app.schemas.reasoning import ReasoningResult


class DisputeReasoner:

    def __init__(
        self,
        *,
        model: StructuredGenerator,
        prompt_loader: PromptLoader,
        category_loader: CategoryLoader,
    ):
        self._model = model
        self._prompt_loader = prompt_loader
        self._category_loader = category_loader

    async def reason(
        self,
        *,
        case_context: str,
        network: CardNetwork,
        language: Language,
        classification: ClassificationResult,
        policy_context: str,
    ) -> ReasoningResult:

        system_prompt = self._prompt_loader.load_system(
            prompt_type="reasoning",
        )

        task_template = self._prompt_loader.load_task(
            prompt_type="reasoning",
            network=network,
            language=language,
        )

        category = self._category_loader.get(
            network=network,
            language=language,
            category_key=classification.category,
        )

        user_prompt = task_template.format(
            case_context=case_context,
            category=classification.category,
            category_display_name=category.display_name,
            category_description=category.description,
            policy_context=policy_context,
        )

        return await self._model.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            response_model=ReasoningResult,
        )

In [ ]:
# app/workflows/dispute_workflow.py

class DisputeWorkflow:

    def __init__(
        self,
        *,
        classifier,
        retriever,
        reasoner,
        case_context_builder,
    ):
        self._classifier = classifier
        self._retriever = retriever
        self._reasoner = reasoner
        self._case_context_builder = case_context_builder

    async def process(self, request):

        case_context = self._case_context_builder.build(
            case_data=request.caseData,
            crsqa=request.CRSQA,
            summary_notes=request.summaryNotes,
            fraud_non_fraud_qa=request.fraudNonFraudQA,
        )

        network = request.caseData.caseType
        language = request.caseData.language

        classification = await self._classifier.classify(
            case_context=case_context,
            network=network,
            language=language,
        )

        policy_context = await self._retriever.retrieve(
            network=network,
            category=classification.category,
            case_context=case_context,
        )

        reasoning = await self._reasoner.reason(
            case_context=case_context,
            network=network,
            language=language,
            classification=classification,
            policy_context=policy_context,
        )

        return {
            "classification": classification,
            "reasoning": reasoning,
        }